# Day 01 — Why Factor Models Fail: The Epistemological Problem

**HongJin HE · Alpha Flow Research Notes · July 2026**

---

## The Question

I had a simple realization. Factor models have been the dominant paradigm in quantitative finance for 50 years. The Fama-French three-factor model. APT. Risk parity. Smart beta. All of these share a common assumption that I think is fundamentally wrong.

The question is: **are we doing science or are we doing statistics?**

Because those are different things.

## The Naive View: Factors Are the Answer

The standard factor model looks like this:

$$r_t = \alpha + \beta_1 f_{1,t} + \beta_2 f_{2,t} + \cdots + \beta_k f_{k,t} + \epsilon_t$$

You pick some factors $f_i$ — market return, size, value, momentum — fit the betas on historical data, and use them to predict future returns or build optimal portfolios.

This paradigm answers: **"what goes up when X goes up?"**

It does not answer: **"why does X go up?"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)
T = 500

# Generate a 'factor' and a correlated return series
factor = np.random.randn(T)
# Two different DATA GENERATING PROCESSES that produce the same correlation

# Process 1: stable causal relationship (beta is constant)
beta_true = 0.6
returns_stable = beta_true * factor + 0.3 * np.random.randn(T)

# Process 2: regime-switching relationship (same average correlation, but unstable)
regime = np.sin(np.linspace(0, 4*np.pi, T)) > 0
beta_regime = np.where(regime, 1.2, -0.05)  # strong positive vs. near-zero
returns_regime = beta_regime * factor + 0.3 * np.random.randn(T)

# Both have similar IN-SAMPLE R^2
r2_stable = stats.pearsonr(factor, returns_stable)[0]**2
r2_regime = stats.pearsonr(factor, returns_regime)[0]**2
print(f"Stable process R²: {r2_stable:.3f}")
print(f"Regime process R²: {r2_regime:.3f}")
print()
print("Factor models cannot distinguish between these two.")
print("One will predict well out-of-sample. The other will blow up.")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

axes[0].scatter(factor[:250], returns_stable[:250], alpha=0.3, label='Train', s=10)
axes[0].scatter(factor[250:], returns_stable[250:], alpha=0.3, label='Test', s=10, color='red')
axes[0].set_title('Stable process: same beta in-sample and out-of-sample')
axes[0].legend()

axes[1].scatter(factor[:250], returns_regime[:250], alpha=0.3, label='Train', s=10)
axes[1].scatter(factor[250:], returns_regime[250:], alpha=0.3, label='Test', s=10, color='red')
axes[1].set_title('Regime process: betas shift — OOS predictions become wrong')
axes[1].legend()

plt.tight_layout()
plt.savefig('day01_factor_instability.png', dpi=150)
plt.show()

## Why It Breaks: The Correlation Is Not the Mechanism

Here's the core problem. Factor models find correlations in historical data. But a correlation between variable A and variable B can arise from:

1. A causes B (genuine causal mechanism)
2. B causes A (reverse causation)
3. C causes both A and B (confounding)
4. Pure chance in the sample (spurious correlation)

Factor models **cannot distinguish** between these four cases. They all look the same in a regression.

This matters because:
- Cases 1 and 2: relationship might persist out-of-sample
- Case 3: relationship breaks when C changes
- Case 4: relationship definitely breaks out-of-sample

The factor model only asks "how correlated were they historically?" — it does not ask "will this mechanism persist?"

**Yann LeCun's critique of LLMs applies here.** LeCun says LLMs are "text probability fitters" — they learn statistical patterns in text without understanding the causal structure of the world. Factor models are the same: they are **return probability fitters**, learning patterns in price history without understanding the causal structure of markets.

## First Principles: What Would a Market Model That Understands Causation Look Like?

Let me think from scratch. What actually determines a stock price at time $t$?

**Naive answer:** supply and demand.

**Better answer:** supply and demand are themselves determined by:
- Agents' beliefs about future cash flows
- Agents' beliefs about other agents' beliefs (Keynes beauty contest)
- Risk appetite of marginal investors
- Liquidity constraints
- External information arrivals (earnings, macro, geopolitics)

**Better still:** all of this is a dynamical system. The state at $t+1$ is a function of the state at $t$ plus new information plus the game being played between agents.

The proper mathematical object is not:
$$r_t = \alpha + \beta f_t + \epsilon_t \quad \text{(factor model)}$$

But rather:
$$X_{t+\Delta t} = X_t + \mu(X_t, t)\Delta t + \sigma(X_t, t) \Delta W_t \quad \text{(dynamical system)}$$

Where $X_t$ is the full **state** of the financial world — not just prices, but agent beliefs, liquidity, macro regime, sentiment — and $W_t$ is the randomness that cannot be predicted.

The question becomes: **can we learn $\mu(\cdot)$ and $\sigma(\cdot)$?** Can we learn the functions that govern how the state evolves?

This is the world model approach.

In [ ]:
# Illustrate the difference between factor model and dynamical model
# For a simple mean-reverting process (Ornstein-Uhlenbeck)

dt = 0.01
T_steps = 2000
mu_rev = 0.5  # mean-reversion speed
theta = 0.0   # long-run mean
sigma = 0.3

# True dynamical model: dX = mu_rev*(theta - X)*dt + sigma*dW
X = np.zeros(T_steps)
X[0] = 1.0
for t in range(1, T_steps):
    X[t] = X[t-1] + mu_rev * (theta - X[t-1]) * dt + sigma * np.sqrt(dt) * np.random.randn()

# What a factor model would try to do: regress X[t] on X[t-1] (or lags)
# This captures SOME of the dynamics, but not the nonlinear state-dependence
lag1 = X[:-1]
curr = X[1:]
slope, intercept, r_val, _, _ = stats.linregress(lag1, curr)
print(f"Factor model (lag-1 regression): slope={slope:.3f}, R²={r_val**2:.3f}")
print(f"True drift at X=1.0: {mu_rev*(theta-1.0)*dt:.4f}")
print(f"True drift at X=-1.0: {mu_rev*(theta-(-1.0))*dt:.4f}")
print()
print("A factor model fits one global slope — it cannot see state-dependent drift.")
print("A world model learns mu(X) at every state X.")

## The Key Insight

Factor models look **backward** — "what patterns existed in historical data?"

World models look **forward** — "what is the mechanism that generates future states?"

This is not just a philosophical distinction. It has a concrete mathematical consequence:

- Factor models assume betas are **constant** (or slowly varying)
- The actual financial world has **state-dependent** dynamics

When the state changes (regime shift, liquidity crisis, structural break), factor model betas become wrong. The model has memorized correlation without understanding causation.

A world model, if correctly specified, should **extrapolate correctly to new states** because it has learned the mechanism, not just the history.

---

**Tomorrow:** If the world model should learn a dynamical system, why do we use stochastic differential equations instead of ordinary differential equations? The answer involves why noise in financial markets is fundamentally different from noise in physics.

*(Day 02: The Roughness of Market Noise)*